# ELEC 475 Lab 2 — Colab Runner

Runs the Lab 2 experiments on a Colab GPU. Same commands as `commands.txt`.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. Your repo is private, so add a GitHub token as a Colab secret:
   - Make a token at https://github.com/settings/personal-access-tokens/new (fine-grained, *Only select repositories* → `ELEC_475_Lab2`, *Repository permissions → Contents: Read-only*).
   - In Colab, click the **key icon** (Secrets) in the left sidebar → **Add new secret** → name `GITHUB_TOKEN`, paste the token, and turn on **Notebook access**.
3. Push your latest commits to GitHub first — Colab clones what's on GitHub, not your laptop.

Run the setup cells (1–4), then only the experiment cells you need. Run cell **Save** at the end — Colab deletes everything in `/content` when the session ends.

## 1. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive (where results are saved)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/ELEC475/lab2_results'

## 3. Clone (or update) the repo

In [ ]:
import os
from google.colab import userdata

REPO = 'geoffrey329/ELEC_475_Lab2'
BRANCH = 'main'
WORKDIR = '/content/ELEC_475_Lab2'

token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{REPO}.git'

if os.path.isdir(WORKDIR):
    !git -C {WORKDIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {url} {WORKDIR}

%cd {WORKDIR}
!git log --oneline -3

## 4. Install requirements

In [ ]:
# torch/torchvision are preinstalled on Colab with CUDA; this adds the rest.
!pip install -q -r requirements.txt
!mkdir -p logs plots checkpoints data
!python -m models.alexnet 2>&1 | tail -5

---
## Pre-Lab: verification, EDA, transforms demo

In [ ]:
!python prelab_verify.py
!python -m utils.eda --dataset cifar10
!python -m utils.eda --dataset tiny_imagenet
!python demo_transforms.py

## In-Lab: LeNet-5 and Modernized LeNet (5 epochs, CIFAR-10)

In [ ]:
# Log names match what plot_ablation.py reads (train_lenet_t4.log / train_lenet_modern_t4.log)
!python train.py --model lenet --dataset cifar10 --epochs 5 --seed 42 --no-augment --save-model checkpoints/lenet_baseline.pth --plot-file plots/loss_lenet.png 2>&1 | tee logs/train_lenet_t4.log
!python train.py --model lenet_modern --dataset cifar10 --epochs 5 --seed 42 --no-augment --save-model checkpoints/lenet_modern.pth --plot-file plots/loss_lenet_modern.png 2>&1 | tee logs/train_lenet_modern_t4.log

## Step 2: Normalization study (10 epochs, CIFAR-10, no augmentation)

In [ ]:
!python train.py --model alexnet --norm none --dataset cifar10 --epochs 10 --seed 42 --no-augment --save-model checkpoints/alexnet_none.pth --plot-file plots/loss_alexnet_none.png 2>&1 | tee logs/train_alexnet_none.log
!python train.py --model alexnet --norm lrn --dataset cifar10 --epochs 10 --seed 42 --no-augment --save-model checkpoints/alexnet_lrn.pth --plot-file plots/loss_alexnet_lrn.png 2>&1 | tee logs/train_alexnet_lrn.log
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --seed 42 --no-augment --save-model checkpoints/alexnet_bn.pth --plot-file plots/loss_alexnet_bn.png 2>&1 | tee logs/train_alexnet_bn.log

## Step 3: Data augmentation (15 epochs, CIFAR-10)

In [ ]:
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 15 --seed 42 --no-augment --save-model checkpoints/alexnet_no_aug.pth --plot-file plots/loss_alexnet_no_aug.png 2>&1 | tee logs/train_alexnet_no_aug.log
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 15 --seed 42 --save-model checkpoints/alexnet_with_aug.pth --plot-file plots/loss_alexnet_with_aug.png 2>&1 | tee logs/train_alexnet_with_aug.log

## Step 4: Tiny ImageNet-200
The first run downloads the dataset (~240 MB).

In [ ]:
!python train.py --model alexnet --norm bn --dataset tiny_imagenet --epochs 15 --seed 42 --save-model checkpoints/alexnet_tiny_imagenet.pth --plot-file plots/loss_alexnet_tiny_imagenet.png 2>&1 | tee logs/train_alexnet_tiny_imagenet.log

Remaining Table 7.3b Tiny ImageNet runs:

In [ ]:
!python train.py --model lenet --dataset tiny_imagenet --epochs 5 --seed 42 --no-augment --save-model checkpoints/lenet_tiny.pth --plot-file plots/loss_lenet_tiny.png 2>&1 | tee logs/train_lenet_tiny.log
!python train.py --model lenet_modern --dataset tiny_imagenet --epochs 5 --seed 42 --no-augment --save-model checkpoints/lenet_modern_tiny.pth --plot-file plots/loss_lenet_modern_tiny.png 2>&1 | tee logs/train_lenet_modern_tiny.log
!python train.py --model alexnet --norm none --dataset tiny_imagenet --epochs 10 --seed 42 --no-augment --save-model checkpoints/alexnet_none_tiny.pth --plot-file plots/loss_alexnet_none_tiny.png 2>&1 | tee logs/train_alexnet_none_tiny.log
!python train.py --model alexnet --norm lrn --dataset tiny_imagenet --epochs 10 --seed 42 --no-augment --save-model checkpoints/alexnet_lrn_tiny.pth --plot-file plots/loss_alexnet_lrn_tiny.png 2>&1 | tee logs/train_alexnet_lrn_tiny.log
!python train.py --model alexnet --norm bn --dataset tiny_imagenet --epochs 10 --seed 42 --no-augment --save-model checkpoints/alexnet_bn_tiny.pth --plot-file plots/loss_alexnet_bn_tiny.png 2>&1 | tee logs/train_alexnet_bn_tiny.log
!python train.py --model zoo_alexnet --dataset tiny_imagenet --epochs 10 --seed 42 --no-augment --save-model checkpoints/zoo_alexnet_tiny.pth --plot-file plots/loss_zoo_alexnet_tiny.png 2>&1 | tee logs/train_zoo_alexnet_tiny.log

## Step 5: Zoo baseline, multi-seed, complexity table, ablation plots

In [ ]:
!python train.py --model zoo_alexnet --dataset cifar10 --epochs 10 --seed 42 --no-augment --save-model checkpoints/zoo_alexnet.pth --plot-file plots/loss_zoo_alexnet.png 2>&1 | tee logs/train_zoo_alexnet.log
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --seed 43 --no-augment --save-model checkpoints/alexnet_bn_s43.pth --plot-file plots/loss_alexnet_bn_s43.png 2>&1 | tee logs/train_alexnet_bn_s43.log
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --seed 44 --no-augment --save-model checkpoints/alexnet_bn_s44.pth --plot-file plots/loss_alexnet_bn_s44.png 2>&1 | tee logs/train_alexnet_bn_s44.log

In [ ]:
# The provided profiler fails when a GPU is visible, so hide it for this command.
!CUDA_VISIBLE_DEVICES="" python compare_models.py --dataset cifar10 2>&1 | tee logs/compare_models.log
!python plot_ablation.py
!python plot_ablation.py --dataset tiny_imagenet

---
## Save results to Google Drive
Run this before closing the session.

In [ ]:
!mkdir -p {RESULTS_DIR}
!cp -r logs plots checkpoints {RESULTS_DIR}/
!ls -lh {RESULTS_DIR}/logs | tail -n +1
print('Saved to', RESULTS_DIR)